# 01 — Problem Definition & Dataset Audit

## Objective
Establish the actual dataset schema, data quality, target distribution, temporal coverage, entity availability, IP-country relationship, leakage risks, and a defensible chronological split.

**Scientific principles enforced in this notebook**
- Fraud label (`class`) is **validation ground truth only** — never a training feature.
- We audit leakage risks (e.g. future information, target leakage).
- Chronological train / validation / test split is defined and locked.
- No model training occurs here.

## Research question
> Does modelling e-commerce activity as a dynamic multipartite graph (transactions ↔ users ↔ devices ↔ IPs) surface anomalous behaviour beyond what simple transaction-level statistics can detect?


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_raw, parse_timestamps, enrich_with_country, basic_clean, chronological_split, save_processed

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

fraud, ipmap = load_raw()
print("Fraud_Data shape:", fraud.shape)
print("IpAddress_to_Country shape:", ipmap.shape)
print("\nFraud columns:", fraud.columns.tolist())
print("IP lookup columns:", ipmap.columns.tolist())


## 1. Schema & Data Quality Audit

In [ ]:
print("=== DATA TYPES & MISSINGNESS ===")
quality = pd.DataFrame({
    "dtype": fraud.dtypes.astype(str),
    "non_null": fraud.notna().sum(),
    "missing": fraud.isna().sum(),
    "missing_pct": (fraud.isna().mean() * 100).round(3),
    "nunique": fraud.nunique(dropna=True)
})
display(quality)

print("\n=== DUPLICATES ===")
print("Exact duplicate rows in Fraud_Data:", fraud.duplicated().sum())
print("Exact duplicate rows in IP map:", ipmap.duplicated().sum())

print("\n=== ENTITY CARDINALITY ===")
for c in ["user_id", "device_id", "ip_address"]:
    print(f"{c}: unique={fraud[c].nunique():,}")


## 2. Target Distribution (Fraud Label)

In [ ]:
TARGET = "class"
assert TARGET in fraud.columns

print("Value counts:")
display(fraud[TARGET].value_counts(dropna=False).to_frame("count"))
print("\nProportions:")
display((fraud[TARGET].value_counts(normalize=True) * 100).round(3).to_frame("percent"))

fig, ax = plt.subplots(figsize=(5, 3.5))
fraud[TARGET].value_counts().plot(kind="bar", color=["#4C78A8", "#E45756"], ax=ax)
ax.set_title("Fraud label distribution")
ax.set_xlabel("class (0=normal, 1=fraud)")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()


## 3. Temporal Coverage & Account-Age Signals

In [ ]:
fraud = parse_timestamps(fraud)

for c in ["signup_time", "purchase_time"]:
    print(f"{c}: min={fraud[c].min()}, max={fraud[c].max()}, nulls={fraud[c].isna().sum()}")

delta = fraud["purchase_time"] - fraud["signup_time"]
fraud["account_age_hours"] = delta.dt.total_seconds() / 3600
fraud["account_age_seconds"] = delta.dt.total_seconds()

print("\nAccount age (hours) summary:")
display(fraud["account_age_hours"].describe())

print("\nInstant purchases (≤1 second after signup):", (fraud["account_age_seconds"] <= 1).sum())
print("Same-day purchases (≤24 h):", (fraud["account_age_hours"] <= 24).sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fraud["account_age_hours"].clip(upper=500).hist(bins=50, ax=axes[0], color="#4C78A8")
axes[0].set_title("Account age (hours, clipped at 500)")
axes[0].set_xlabel("hours")

# Fraud rate vs account age buckets
bins = [0, 0.001, 1, 24, 168, 720, np.inf]
labels = ["≤1s", "≤1h", "≤1d", "≤1w", "≤1m", ">1m"]
fraud["age_bucket"] = pd.cut(fraud["account_age_hours"], bins=bins, labels=labels)
rate = fraud.groupby("age_bucket", observed=True)["class"].mean()
rate.plot(kind="bar", ax=axes[1], color="#E45756")
axes[1].set_title("Fraud rate by account-age bucket")
axes[1].set_ylabel("fraud rate")
plt.tight_layout()
plt.show()


## 4. IP → Country Enrichment

In [ ]:
fraud = enrich_with_country(fraud, ipmap)
print("Country coverage:")
display(fraud["country"].value_counts().head(15).to_frame("count"))
print("\nUnknown country rate: {:.2f}%".format((fraud["country"] == "Unknown").mean() * 100))


## 5. Basic Cleaning & Chronological Split

We define a strict time-based split on `purchase_time`:
- **Train** 60 % earliest transactions
- **Validation** next 20 %
- **Test** final 20 % (locked until notebook 11)


In [ ]:
fraud = basic_clean(fraud)
print("After basic clean:", fraud.shape)

train, val, test = chronological_split(fraud, train_frac=0.60, val_frac=0.20)
print(f"Train: {len(train):,}  ({train['purchase_time'].min().date()} → {train['purchase_time'].max().date()})")
print(f"Val:   {len(val):,}  ({val['purchase_time'].min().date()} → {val['purchase_time'].max().date()})")
print(f"Test:  {len(test):,}  ({test['purchase_time'].min().date()} → {test['purchase_time'].max().date()})")

print("\nFraud rate per split:")
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"  {name}: {part['class'].mean()*100:.2f}%")

# Persist for downstream notebooks
save_processed(fraud, "fraud_enriched")
save_processed(train, "train")
save_processed(val, "val")
save_processed(test, "test")
print("\nSaved processed parquet files to data/processed/")


## 6. Leakage Risk Checklist

| Risk | Status | Mitigation |
|------|--------|------------|
| Target leakage | Controlled | `class` never enters feature matrix |
| Temporal leakage | Controlled | Strict chronological split |
| Entity leakage (same user in train & test) | Accepted | Real-world entities persist; we still evaluate ranking quality |
| IP-country map | Safe | Static lookup table, no future info |
| Aggregate statistics computed on full data | Caution | Downstream notebooks recompute features inside each split when required |

## Deliverables of this notebook
- Cleaned & enriched transaction table
- Chronological train / val / test partitions
- Documented data quality profile
- Locked experimental protocol


In [ ]:
# Write a lightweight profile for the reports folder
profile = {
    "n_rows": int(len(fraud)),
    "n_users": int(fraud["user_id"].nunique()),
    "n_devices": int(fraud["device_id"].nunique()),
    "n_ips": int(fraud["ip_address"].nunique()),
    "fraud_rate": float(fraud["class"].mean()),
    "time_span": [str(fraud["purchase_time"].min()), str(fraud["purchase_time"].max())],
    "train_size": int(len(train)),
    "val_size": int(len(val)),
    "test_size": int(len(test)),
}
(Path(ROOT) / "reports" / "initial_data_profile.json").write_text(json.dumps(profile, indent=2))
print("Profile written.")
print(json.dumps(profile, indent=2))
